# Topic F — L6: Variance Reduction for Monte Carlo

*FINM 32000 Numerical Methods — Final-Exam Study Notes*

---

**Prerequisites:** [Topic E — MC Foundations](Topic-E_L5_MonteCarlo_Foundations.ipynb) (MC estimator, SE, confidence intervals, Euler simulation).

**What this notebook covers**
1. Why we need variance reduction (the $1/\sqrt M$ wall)
2. **Antithetic variates** — pairing $Z$ with $-Z$
3. **Control variates** — subtract a correlated, known-mean variable
4. **Conditional Monte Carlo** — integrate one dimension analytically
5. *(Brief)* Importance sampling — **excluded from final exam**, light touch only
6. **Practice-Final P3 walkthrough** — find the 3 bugs in conditional MC code
7. **HW6 P1 walkthrough** — basket option: ordinary vs antithetic vs control variate
8. How to pick the right technique
9. Common pitfalls + exercises + cheat sheet

**Exam map**

| Source | What it tests |
|---|---|
| L6.4–L6.8 | Antithetic variates: bias, variance, when it helps |
| L6.10–L6.15 | Control variates: optimal $\beta$, $\hat\beta$, CI |
| L6.29–L6.36 | Conditional MC: variance bound, "mixing formula" |
| **Practice-Final P3** | Spot bugs in conditional MC code (3 lines) |
| HW6 P1 | Antithetic + control variates on basket call |

**Reminder:** Per the professor, **importance sampling is excluded from the final exam.** We include it briefly for completeness only.


In [1]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=6, suppress=True)

def bs_call(S, K, T, r, sigma, q=0.0):
    if T <= 0:
        return max(S - K, 0.0)
    std = sigma * np.sqrt(T)
    d1 = (np.log(S/K) + (r - q + 0.5*sigma**2)*T) / std
    d2 = d1 - std
    return S*np.exp(-q*T)*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)


## 1. Why variance reduction? (L6.2)

From Topic E, the MC error shrinks like $1/\sqrt M$:
$$\text{radius of 95% CI} = 1.96 \cdot \frac{\hat\sigma}{\sqrt M}.$$

To get one more decimal place, $M$ must grow $100\times$.

**Variance reduction techniques** build estimators with **lower $\hat\sigma$** for the **same** $M$. The expectation stays equal to $C$, but the variance drops.

### Three flavors

| Idea | Technique |
|---|---|
| **Introduce dependence** across simulations | Antithetic variates |
| **Simulate something other than the payoff** | Control variates, Conditional MC |
| **Change the probability distribution** | Importance sampling (*excluded*) |

Increasing $M$ is **not** "variance reduction" — it just costs more computation.


## 2. Antithetic variates (L6.4–L6.8)

### Core idea

If you draw a realization from the $q^{\text{th}}$ percentile, also use $(100-q)^{\text{th}}$ percentile.

Because the standard normal is symmetric, for every $Z_m$ that you draw, you also use $-Z_m$.

### Estimator

For each $m = 1, \ldots, M$:
$$\tilde Z_m := -Z_m, \quad \tilde Y_m := \text{payoff with } \tilde Z_m, \quad Y_m^{\text{av}} := \frac{Y_m + \tilde Y_m}{2}.$$

The antithetic-variate MC estimate:
$$\boxed{\widehat C_M^{\text{av}} := \frac{1}{M}\sum_{m=1}^M Y_m^{\text{av}}.}$$

### Unbiased?

$$\mathbb E[\widehat C_M^{\text{av}}] = \mathbb E\!\left[\frac{Y + \tilde Y}{2}\right] = \frac{\mathbb E Y + \mathbb E\tilde Y}{2} = \mathbb E Y = C. \checkmark$$

### Variance analysis

$$\mathrm{Var}(\widehat C_M^{\text{av}}) = \frac{1}{M}\mathrm{Var}(Y^{\text{av}}) = \frac{1}{M}\cdot \frac{\mathrm{Var}(Y) + \mathrm{Cov}(Y,\tilde Y)}{2}.$$

**Fair comparison** is against ordinary MC with $2M$ paths (same total simulations):
$$\mathrm{Var}(\widehat C_{2M}) = \frac{\mathrm{Var}(Y)}{2M}.$$

Antithetic helps iff
$$\boxed{\mathrm{Cov}(Y, \tilde Y) < 0.}$$

**Plain English:** Antithetic helps when the payoff is roughly **monotonic** in the random driver. For a vanilla call, $Y(Z)$ is increasing in $Z$, so $\mathrm{Cov}(Y(Z), Y(-Z)) < 0$ — antithetic helps. For payoffs that are symmetric or nonmonotonic (e.g. a straddle near ATM), antithetic can **fail to help**.

### Standard error

$$\widehat{\text{SE}}^{\text{av}} = \frac{\hat\sigma_M^{\text{av}}}{\sqrt M}, \qquad (\hat\sigma_M^{\text{av}})^2 = \frac{1}{M-1}\sum_m (Y_m^{\text{av}} - \widehat C_M^{\text{av}})^2.$$

**Critical:** Use the sample standard deviation of the **pair averages** $Y_m^{\text{av}}$, not of the $2M$ individual realizations.


In [2]:
rng = np.random.default_rng(seed=0)
S0, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20
M = 10_000

def price_gbm_call(Z):
    ST = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    return np.exp(-r*T) * np.maximum(ST - K, 0.0)

Z = rng.standard_normal(M)
Y = price_gbm_call(Z)
Y_tilde = price_gbm_call(-Z)
Y_av = 0.5 * (Y + Y_tilde)

mean_ord = Y.mean()
se_ord = Y.std(ddof=1) / np.sqrt(M)

mean_av = Y_av.mean()
se_av = Y_av.std(ddof=1) / np.sqrt(M)

truth = bs_call(S0, K, T, r, sigma)
print(f"BS truth:        {truth:.4f}")
print()
print(f"Ordinary MC  (M={M}): price {mean_ord:.4f}, SE {se_ord:.4f}, 95% radius {1.96*se_ord:.4f}")
print(f"Antithetic   (M={M}): price {mean_av:.4f}, SE {se_av:.4f}, 95% radius {1.96*se_av:.4f}")
print(f"\nCov(Y, Y_tilde): {np.cov(Y, Y_tilde, ddof=1)[0,1]:.4f}  (negative -> AV helps)")
print(f"Variance reduction factor: ~{(se_ord/se_av)**2:.2f}x")


BS truth:        10.4506

Ordinary MC  (M=10000): price 10.5194, SE 0.1462, 95% radius 0.2865
Antithetic   (M=10000): price 10.4557, SE 0.0729, 95% radius 0.1429

Cov(Y, Y_tilde): -108.7084  (negative -> AV helps)
Variance reduction factor: ~4.02x


## 3. Control variates (L6.10–L6.15)

### Core idea

Find a random variable $Y^*$ that
- is **correlated with $Y$** (so simulating it tells you something about $Y$), and
- has **known expectation** $C^* := \mathbb E[Y^*]$ (from a closed-form formula, typically Black-Scholes).

Then **simulate $Y - \beta Y^*$** instead of $Y$.

### Two famous choices

| Original $Y$ | Control $Y^*$ | Why $C^*$ is known |
|---|---|---|
| Discounted Asian payoff on GBM | Discounted European payoff on GBM | Black-Scholes formula |
| Discounted Asian payoff on GBM | Discounted **geometric Asian** payoff | Geometric Asian has closed form |
| Discounted call on stochastic-vol $S$ | Discounted call on constant-vol $S^*$ | Black-Scholes formula |
| Discounted arithmetic basket call | Discounted **geometric basket** call | Geometric basket is lognormal — closed form |

### Estimator (fixed $\beta$)

$$\boxed{\widehat C_M^{\text{cv},\beta} := \bar Y_M + \beta(C^* - \bar Y^*_M).}$$

### Unbiased?

$$\mathbb E[\widehat C_M^{\text{cv},\beta}] = \mathbb E Y + \beta(C^* - \mathbb E Y^*) = C + \beta \cdot 0 = C. \checkmark$$

### Variance and optimal $\beta$

$$\mathrm{Var}(\widehat C_M^{\text{cv},\beta}) = \frac{1}{M}\Bigl[\mathrm{Var}(Y) - 2\beta\,\mathrm{Cov}(Y, Y^*) + \beta^2\mathrm{Var}(Y^*)\Bigr].$$

Differentiate in $\beta$, set to zero:
$$\boxed{\beta_{\text{optimal}} = \frac{\mathrm{Cov}(Y, Y^*)}{\mathrm{Var}(Y^*)}}.$$

At the optimum:
$$\mathrm{Var}(\widehat C_M^{\text{cv}, \beta_{\text{opt}}}) = \frac{1}{M}\mathrm{Var}(Y)\bigl(1 - \mathrm{Corr}^2(Y, Y^*)\bigr).$$

**Plain English:** If you find a control $Y^*$ with $\mathrm{Corr}(Y, Y^*) = 0.99$, the variance shrinks by $1 - 0.99^2 = 0.02$ — a **50× speedup** at the same accuracy. If correlation is 0.9, the speedup is ~5×.

### Using $\hat\beta$ (sample covariance) — L6.13

In practice $\mathrm{Cov}$ and $\mathrm{Var}$ are unknown. Use:
$$\boxed{\hat\beta := \frac{\sum_m (Y_m - \bar Y_M)(Y^*_m - \bar Y^*_M)}{\sum_m (Y^*_m - \bar Y^*_M)^2}.}$$

This is just the OLS slope of $Y$ regressed on $Y^*$.

**Bias note:** Using $\hat\beta$ introduces small-sample bias, but the bias $\to 0$ as $M \to \infty$, so confidence intervals are asymptotically valid.

### Standard error (L6.14–L6.15)

Form the **per-path control variate values**:
$$Y_m^{\text{cv},\beta} := Y_m + \beta(C^* - Y^*_m).$$
Then
$$\widehat C_M^{\text{cv},\beta} = \frac{1}{M}\sum Y_m^{\text{cv},\beta},\qquad
\widehat{\text{SE}}^{\text{cv},\beta} = \frac{\hat\sigma^{\text{cv},\beta}_M}{\sqrt M}.$$

Replace $\beta$ with $\hat\beta$ for the practical estimator.


## 4. Conditional Monte Carlo (L6.29–L6.36)

### The trick — tower property

By the law of iterated expectations (Topic 0 §10):
$$C = \mathbb E[Y] = \mathbb E\bigl[\mathbb E[Y \mid X]\bigr] = \mathbb E[f(X)]$$
where $f(x) := \mathbb E[Y \mid X = x]$.

**If you can compute $f(X)$ analytically**, then instead of simulating $Y$ you simulate $X$ and average $f(X)$.

### Algorithm

1. Draw $X_1, \ldots, X_M$ IID like $X$.
2. Compute $f(X_m)$ analytically (typically a Black-Scholes formula).
3. Average: $\widehat C_M^{\text{cmc}} = \frac{1}{M}\sum_m f(X_m)$.

### Variance bound

$$\mathrm{Var}(Y) = \mathrm{Var}\bigl[\mathbb E(Y\mid X)\bigr] + \mathbb E\bigl[\mathrm{Var}(Y \mid X)\bigr].$$

So
$$\boxed{\mathrm{Var}(\widehat C_M^{\text{cmc}}) = \frac{1}{M}\mathrm{Var}(f(X)) \le \frac{1}{M}\mathrm{Var}(Y) = \mathrm{Var}(\widehat C_M).}$$

**CMC always reduces variance.** The bigger the noise inside the conditional expectation (i.e., the bigger $\mathbb E\mathrm{Var}(Y|X)$), the bigger the speedup.

### Intuition

If $Y = F(X)$ where $X \in \mathbb R^D$, ordinary MC is a $D$-dim numerical integration. If $d$ of those dimensions can be integrated **analytically**, we only need to draw $X_m \in \mathbb R^{D-d}$. Fewer dimensions of randomness $\Rightarrow$ less noise.

### Example 1 — Asian-style call (L6.31)

Payoff: $Y = e^{-rt_3}\bigl(\tfrac13(S_{t_1} + S_{t_2} + S_{t_3}) - k\bigr)^+$ under GBM.

Conditional on $X = (S_{t_1}, S_{t_2})$:
$$Y = \tfrac{1}{3}e^{-rt_3}\bigl(S_{t_3} - (3k - S_{t_1} - S_{t_2})\bigr)^+ \cdot 1.$$

Conditional on $S_{t_2}$ and $S_{t_1}$, this is a vanilla European call on GBM starting at $S_{t_2}$ with **shifted strike** $3k - S_{t_1} - S_{t_2}$ and time to expiry $t_3 - t_2$. So
$$\mathbb E[Y \mid S_{t_1}, S_{t_2}] = \tfrac13 e^{-rt_2}\,C_{BS}\bigl(S_{t_2},\; 3k - S_{t_1} - S_{t_2},\; t_3 - t_2,\; r,\; \sigma\bigr).$$

CMC: simulate the pair $(S_{t_1}, S_{t_2})$, plug into BS — never simulate $S_{t_3}$.

### Example 2 — Stochastic vol "mixing formula" (L6.32–L6.36)

For $dS_t = rS_t\,dt + \sqrt{V_t}S_t\,dW_{1,t}$ with $V$ independent of $W_1$:

Conditional on the **entire $V$ path**, $S$ is lognormal with effective volatility
$$\bar\sigma := \sqrt{\frac{1}{T}\int_0^T V_t\,dt}.$$

So the call price is
$$C = \mathbb E\bigl[C_{BS}(\bar\sigma)\bigr].$$

**CMC**: simulate only the $V$ path (1D), compute $C_{BS}(\bar\sigma)$, average. Avoid simulating $S$ entirely.

With correlation $\rho \neq 0$ between $W_1, W_2$, the formula generalizes to "mixing formula" $C = \mathbb E\,C_{BS}(S_0 M_T(\rho),\; \bar\sigma\sqrt{1-\rho^2})$ where $M_T(\rho)$ absorbs the correlated drift.


## 5. Importance sampling — *light touch only*

> **Excluded from the final exam.** Read for context.

### Idea (L6.18–L6.22)

Sample $X$ from a **different** density $g$ (rather than its natural $f$), and weight by the **Radon-Nikodym ratio** $f/g$:
$$\widehat C^{\text{is}} = \frac{1}{M}\sum_m h(X_m)\,\frac{f(X_m)}{g(X_m)},\qquad X_m \sim g.$$

Always unbiased. Variance is minimized when $g \propto h \cdot f$.

### Why it helps "rare events"

For a deep OTM call, ordinary MC wastes paths where $S_T < K$ (payoff is zero). IS shifts the distribution so $S_T > K$ happens more often, then corrects with the likelihood ratio.

### Why we skip details

Three reasons:
1. The professor said IS is **not on the final**.
2. The bug-spotting question (P3) is **conditional MC**, not IS.
3. IS is delicate (variance can **increase** if $g$ is poorly chosen).

If interested for general knowledge: changing BM drift to $\lambda$ multiplies the payoff by $e^{-\lambda W_T + \lambda^2 T / 2}$ — Girsanov in MC dress (L6.25).


## 6. Practice-Final P3 walkthrough — 3 bugs to find

### The problem

Price a 2-observation Asian-style call on GBM:
$$\text{Payoff at } T_2: \quad \left(\frac{S_{T_1} + S_{T_2}}{2} - K\right)^+$$
using **conditional MC** with $X = S_{T_1}$. Given $S_{T_1}$, the payoff is a vanilla European call on $S$ from $T_1 \to T_2$, with shifted strike $2K - S_{T_1}$ — same recipe as L6.31.

The supplied code has **2 mistakes** in `price_AsianCall` and **1 mistake** in the calling notebook cell. From the solutions:

### Bug 1 — missing factor of $1/2$ in the BS line

The conditional expectation formula gives:
$$\mathbb E[Y \mid S_{T_1}] = \tfrac{1}{2}\,e^{-rT_1}\,C_{BS}(S_{T_1},\; 2K - S_{T_1},\; T_2 - T_1,\; r,\; \sigma).$$

**Wrong code:**
```python
results[i] = np.exp(-r*T1) * analyticEngine.BSpriceCall(...)
```

**Correct:**
```python
results[i] = 0.5 * np.exp(-r*T1) * analyticEngine.BSpriceCall(...)
```

**Why:** The arithmetic average $(S_{T_1}+S_{T_2})/2$ produces a factor of $1/2$ inside the call payoff when you pull $S_{T_1}/2$ outside the max. Forgetting this halves the answer (literally).

### Bug 2 — standard error not divided by $\sqrt M$

**Wrong:** `return (np.mean(results), np.std(results, ddof=1))`

**Correct:** `return (np.mean(results), np.std(results, ddof=1) / np.sqrt(self.M))`

**Why:** What's returned should be the **standard error of the estimator** $\hat\sigma_M/\sqrt M$, not the sample std of one realization. Practice-Final P3 specifically asks for the CI **radius** at the end.

### Bug 3 — wrong drift (dividends!)

In the calling cell:
```python
pf3dynamics = GBM(sigma=0.7, drift=0.03, S_t=10, t=0, r=0.03)
```

The problem says the stock **pays continuous dividends at rate $q = 0.01$**. The risk-neutral drift is $r - q = 0.02$, not $r = 0.03$.

**Correct:**
```python
pf3dynamics = GBM(sigma=0.7, drift=0.02, S_t=10, t=0, r=0.03)
```

### Exam takeaways

1. **Read the problem for dividends.** They flow into the drift, not the discount factor.
2. **Standard error vs sample std** — always divide by $\sqrt M$.
3. **Constants outside the max** — when you pull factors out of a payoff to use a closed-form, keep track of the multiplicative constants.

### Reference: a working version

The cell below implements the **fixed** conditional MC for the 2-period Asian.


In [3]:
class GBM_PF3:
    def __init__(self, sigma, drift, S_t, t, r):
        self.sigma, self.drift = sigma, drift
        self.S_t, self.t, self.r = S_t, t, r

class CallOption_PF3:
    def __init__(self, K, T):
        self.K, self.T = K, T

class AsianCall_PF3:
    def __init__(self, K, T1, T2):
        self.K, self.T1, self.T2 = K, T1, T2

class AnalyticEngine_PF3:
    def BSpriceCall(self, dynamics, contract):
        tau = contract.T - dynamics.t
        F = dynamics.S_t * np.exp(dynamics.drift * tau)
        std = dynamics.sigma * np.sqrt(tau)
        d1 = np.log(F/contract.K)/std + std/2
        d2 = d1 - std
        return np.exp(-dynamics.r*tau)*(F*norm.cdf(d1) - contract.K*norm.cdf(d2))

class MCengine_PF3:
    def __init__(self, M, seed):
        self.M = M
        self.rng = np.random.default_rng(seed=seed)

    def price_AsianCall(self, contract, dynamics):
        engine = AnalyticEngine_PF3()
        T1, T2 = contract.T1, contract.T2
        S0, sigma, drift, r = dynamics.S_t, dynamics.sigma, dynamics.drift, dynamics.r

        from copy import copy
        dyn_T1 = copy(dynamics)
        dyn_T1.t = T1

        results = np.zeros(self.M)
        for i in range(self.M):
            S_T1 = S0 * np.exp((drift - sigma**2/2)*T1 + sigma*np.sqrt(T1)*self.rng.normal())
            dyn_T1.S_t = S_T1
            Kadj = 2*contract.K - S_T1
            if Kadj > 0:
                equiv_call = CallOption_PF3(K=Kadj, T=T2)
                results[i] = 0.5 * np.exp(-r*T1) * engine.BSpriceCall(dyn_T1, equiv_call)
            else:
                results[i] = np.exp(-r*T2) * (S_T1*(1 + np.exp(drift*(T2-T1)))/2 - contract.K)
        return np.mean(results), np.std(results, ddof=1) / np.sqrt(self.M)

contract = AsianCall_PF3(K=12, T1=0.5, T2=1.0)
dyn_correct = GBM_PF3(sigma=0.7, drift=0.02, S_t=10, t=0, r=0.03)  # drift = r - q
dyn_wrong   = GBM_PF3(sigma=0.7, drift=0.03, S_t=10, t=0, r=0.03)  # bug 3

engine = MCengine_PF3(M=10_000, seed=0)
price_ok, se_ok = engine.price_AsianCall(contract, dyn_correct)

engine2 = MCengine_PF3(M=10_000, seed=0)
price_bug, se_bug = engine2.price_AsianCall(contract, dyn_wrong)

z = norm.ppf(0.975)
print(f"Fixed code (drift=0.02): price {price_ok:.4f}, SE {se_ok:.4f}, 95% radius {z*se_ok:.4f}")
print(f"Bug 3 only (drift=0.03): price {price_bug:.4f}  <- biased high")
print(f"\nNote: the printed std is now the standard error (bug 2 fixed)")
print(f"      and the factor 0.5 multiplies BS (bug 1 fixed).")


Fixed code (drift=0.02): price 1.5546, SE 0.0318, 95% radius 0.0623
Bug 3 only (drift=0.03): price 1.5923  <- biased high

Note: the printed std is now the standard error (bug 2 fixed)
      and the factor 0.5 multiplies BS (bug 1 fixed).


## 7. HW6 P1 walkthrough — basket option: ordinary vs antithetic vs control variate

### Setup

Two correlated GBM stocks:
$$dS^{[j]}_t = r S^{[j]}_t\,dt + \sigma^{[j]}S^{[j]}_t\,dW^{[j]}_t, \quad j=1,2$$
with $r=0.05$, $\sigma^{[1]}=0.3$, $\sigma^{[2]}=0.2$, $\rho = 0.8$, $S^{[1]}_0=100$, $S^{[2]}_0=110$.

Contract: arithmetic basket call $(H_T - 110)^+$ where $H_T = \tfrac12 S^{[1]}_T + \tfrac12 S^{[2]}_T$, $T=1$.

### Why arithmetic baskets need MC

The sum of two lognormals is **not** lognormal — no closed form for an arithmetic basket call. We must use MC.

### Step 1 — covariance matrix of log prices

$X_T^{[j]} = \log S_0^{[j]} + (r - \tfrac12\sigma_j^2)T + \sigma_j W^{[j]}_T$. The random part is $\Sigma W_T$, so:
$$\mathrm{Cov}(X_T) = T\,\Sigma\,H\,\Sigma^\top = T\begin{pmatrix}\sigma_1^2 & \rho\sigma_1\sigma_2\\ \rho\sigma_1\sigma_2 & \sigma_2^2\end{pmatrix} = \begin{pmatrix}0.09 & 0.048\\ 0.048 & 0.04\end{pmatrix}.$$

### Step 2 — choose a control variate: the geometric basket

The **geometric basket** $G_T = (S^{[1]}_T S^{[2]}_T)^{1/2}$ is **lognormal** (linear combo of normals stays normal in log-space), so $C_G$ has a closed-form Black-Scholes price.

Match parameters (HW6 P1(d,e)):
$$S_0^G = \sqrt{S_0^{[1]} S_0^{[2]}}, \quad
\sigma_G = \tfrac12\sqrt{\sigma_1^2 + 2\rho\sigma_1\sigma_2 + \sigma_2^2}, \quad
q_G = \tfrac{\sigma_1^2 + \sigma_2^2 - 2\rho\sigma_1\sigma_2}{8}.$$

Then $C_G = C_{BS}(S_0^G,\,K,\,T,\,r-q_G,\,\sigma_G)$ (with dividend yield $q_G$, see Topic 0 §15).

**Why this works as a control:** $G_T$ tracks $H_T$ extremely closely (AM-GM inequality: $G \le H$, but they move together), so $\mathrm{Corr}(Y_H, Y_G) \approx 0.99$. By the variance formula, that's a ~$1/(1-0.99^2) \approx 50\times$ speedup.

### Step 3 — code it up


In [4]:
class MultiGBM:
    def __init__(self, S0, r, correlations, sigma):
        self.S0, self.r = S0, r
        self.correlations = correlations
        self.sigma = sigma  # diag matrix of vols

class CallOnBasket:
    def __init__(self, K, T, weights):
        self.K, self.T, self.weights = K, T, weights

def CBS_q(S, t, K, T, q, r, sigma):
    tau = T - t
    d1 = (np.log(S/K) + (r - q + 0.5*sigma**2)*tau) / (sigma*np.sqrt(tau))
    d2 = d1 - sigma*np.sqrt(tau)
    return S*np.exp(-q*tau)*norm.cdf(d1) - K*np.exp(-r*tau)*norm.cdf(d2)

class HW6MC:
    def __init__(self, M, antithetic, control, seed):
        self.M = M
        self.antithetic = antithetic
        self.control = control
        self.rng = np.random.default_rng(seed=seed)

    def price(self, contract, dynamics):
        S0, r, sigma, corr = dynamics.S0, dynamics.r, dynamics.sigma, dynamics.correlations
        K, T, w = contract.K, contract.T, contract.weights
        M = self.M

        vols = np.diag(sigma)
        cov = T * (sigma @ corr @ sigma)
        mu = np.log(S0) + (r - 0.5*vols**2)*T

        Z = self.rng.multivariate_normal(np.zeros(len(S0)), cov, size=M)
        X = mu + Z
        S = np.exp(X)
        H = S @ w
        Y = np.exp(-r*T) * np.maximum(H - K, 0.0)

        if self.antithetic:
            X_a = mu - Z
            S_a = np.exp(X_a)
            H_a = S_a @ w
            Y_a = np.exp(-r*T) * np.maximum(H_a - K, 0.0)
            Y_av = 0.5*(Y + Y_a)
            return Y_av.mean(), Y_av.std(ddof=1)/np.sqrt(M)

        if self.control:
            G = np.exp(X @ w)
            Ystar = np.exp(-r*T) * np.maximum(G - K, 0.0)

            s1, s2, rho = vols[0], vols[1], corr[0,1]
            sG = 0.5*np.sqrt(s1**2 + 2*rho*s1*s2 + s2**2)
            qG = (s1**2 + s2**2 - 2*rho*s1*s2) / 8.0
            S0G = np.sqrt(S0[0]*S0[1])
            Cstar = CBS_q(S0G, 0.0, K, T, qG, r, sG)

            beta = np.cov(Y, Ystar, ddof=1)[0,1] / np.var(Ystar, ddof=1)
            Y_cv = Y + beta*(Cstar - Ystar)
            return Y_cv.mean(), Y_cv.std(ddof=1)/np.sqrt(M)

        return Y.mean(), Y.std(ddof=1)/np.sqrt(M)

dynamics = MultiGBM(S0=np.array([100,110]), r=0.05,
                    correlations=np.array([[1, 0.8],[0.8, 1]]),
                    sigma=np.diag([0.3, 0.2]))
contract = CallOnBasket(K=110, T=1.0, weights=np.array([0.5, 0.5]))

ord_mc  = HW6MC(M=10_000, antithetic=False, control=False, seed=0)
av_mc   = HW6MC(M=10_000, antithetic=True,  control=False, seed=0)
cv_mc   = HW6MC(M=10_000, antithetic=False, control=True,  seed=0)

p_ord, se_ord = ord_mc.price(contract, dynamics)
p_av,  se_av  = av_mc.price(contract,  dynamics)
p_cv,  se_cv  = cv_mc.price(contract,  dynamics)

print(f"Ordinary MC :  price {p_ord:.4f}, SE {se_ord:.4f}, 95% radius {1.96*se_ord:.4f}")
print(f"Antithetic  :  price {p_av:.4f}, SE {se_av:.4f}, 95% radius {1.96*se_av:.4f}")
print(f"Control var :  price {p_cv:.4f}, SE {se_cv:.4f}, 95% radius {1.96*se_cv:.4f}")
print()
print(f"AV speedup vs ordinary:  ~{(se_ord/se_av)**2:.2f}x variance reduction")
print(f"CV speedup vs ordinary:  ~{(se_ord/se_cv)**2:.2f}x variance reduction  <- big winner")


Ordinary MC :  price 9.8750, SE 0.1684, 95% radius 0.3300
Antithetic  :  price 9.9411, SE 0.0952, 95% radius 0.1866
Control var :  price 9.9910, SE 0.0044, 95% radius 0.0086

AV speedup vs ordinary:  ~3.13x variance reduction
CV speedup vs ordinary:  ~1463.94x variance reduction  <- big winner


## 8. How to pick the right technique

```mermaid
flowchart TD
    A[Need lower MC variance] --> B{Can I integrate out one factor analytically?}
    B -->|Yes - e.g. stoch vol with W1 conditional on V| C[Conditional MC]
    B -->|No| D{Is there a similar contract with known price?}
    D -->|Yes - e.g. geometric basket for arithmetic basket| E[Control variate]
    D -->|No| F{Is payoff roughly monotone in Z?}
    F -->|Yes - vanilla call| G[Antithetic variates]
    F -->|No - straddle, symmetric| H[Antithetic may not help; consider stratification]
```

### Rules of thumb

- **Conditional MC always reduces variance** (when applicable). Look for "I can compute $\mathbb E[Y \mid X]$ in closed form."
- **Control variates** scale with $\mathrm{Corr}^2(Y, Y^*)$. Geometric vs arithmetic basket is a classic 50× speedup.
- **Antithetic variates** are nearly free to add but only help for monotonic payoffs. Often used as a "warm-up" reduction.
- **Stack them**: antithetic + control variate works well together.


## 9. Common pitfalls

### Pitfall 1 — comparing antithetic to MC with $M$ paths

The fair comparison is against ordinary MC with **$2M$ paths**, because antithetic uses $2M$ payoff evaluations. Comparing to MC with $M$ paths overstates the benefit.

### Pitfall 2 — using sample std of the $2M$ realizations

For antithetic, the SE is computed from the $M$ **pair averages** $Y_m^{\text{av}}$, not the $2M$ underlying $Y_m, \tilde Y_m$. Mixing these up overstates variance reduction.

### Pitfall 3 — using a different RNG seed for $Y$ and $Y^*$

Control variates need $Y$ and $Y^*$ from the **same** simulated paths. If you redraw $Z$, the correlation drops and the variance reduction vanishes.

### Pitfall 4 — $\hat\beta$ bias

Using $\hat\beta$ instead of true $\beta$ introduces small-sample bias. Negligible for $M$ in the thousands.

### Pitfall 5 — CMC failure when $\mathbb E[Y \mid X]$ has **no closed form**

If you can't compute $f(X) = \mathbb E[Y \mid X]$ analytically, CMC isn't usable. Don't try to estimate $f$ with a nested MC — that introduces simulation bias.

### Pitfall 6 — forgetting dividends ($r$ vs $r-q$)

Practice-Final P3 Bug 3. Always check the problem statement for dividend yield.

### Pitfall 7 — printing std vs SE

The exam wants the **standard error** ($\hat\sigma_M/\sqrt M$) or the **radius** ($1.96\,\hat\sigma_M/\sqrt M$), not the sample std.


## 10. Exam-style exercises

**E1.** Antithetic variates: under what condition on $\mathrm{Cov}(Y, \tilde Y)$ does antithetic beat ordinary MC with $2M$ paths?

**E2.** A control variate $Y^*$ has $\mathrm{Corr}(Y, Y^*) = 0.95$. What's the variance reduction factor (at optimal $\beta$)?

**E3.** Why is conditional MC **always** lower variance than ordinary MC? Cite the identity.

**E4.** For a vanilla call on GBM, would you use antithetic variates? Why or why not?

**E5.** For a call on stochastic-vol $S$ with independent $V$ and $W_1$, write the conditional MC estimator (mixing formula).


### Solutions

<details><summary><b>E1</b></summary>

$\mathrm{Cov}(Y, \tilde Y) < 0$. Then $\mathrm{Var}(Y^{\text{av}}) = \tfrac12\mathrm{Var}(Y) + \tfrac12\mathrm{Cov}(Y,\tilde Y) < \tfrac12\mathrm{Var}(Y)$, beating ordinary MC with $2M$ paths.
</details>

<details><summary><b>E2</b></summary>

Factor is $1 - 0.95^2 = 0.0975$, i.e. variance shrinks to ~10% of ordinary — a $\sim 10\times$ speedup.
</details>

<details><summary><b>E3</b></summary>

Law of total variance: $\mathrm{Var}(Y) = \mathrm{Var}[\mathbb E(Y|X)] + \mathbb E[\mathrm{Var}(Y|X)] \ge \mathrm{Var}[\mathbb E(Y|X)] = \mathrm{Var}(f(X))$. So $\mathrm{Var}(f(X)) \le \mathrm{Var}(Y)$.
</details>

<details><summary><b>E4</b></summary>

Yes. The payoff $(S_T - K)^+ = (S_0\exp(\nu T + \sigma\sqrt T Z) - K)^+$ is **monotonically increasing in $Z$**. So pairing $Z$ with $-Z$ gives $\mathrm{Cov}(Y, \tilde Y) < 0$, and antithetic reduces variance.
</details>

<details><summary><b>E5</b></summary>

$$\widehat C^{\text{cmc}} = \frac{1}{M}\sum_{m=1}^M C_{BS}\bigl(\bar\sigma_m\bigr),\qquad \bar\sigma_m := \sqrt{\frac{1}{T}\int_0^T V_t^{(m)}\,dt}.$$
Simulate only the $V$ path; never simulate $S$.
</details>


## 11. Cheat sheet — Topic F

| # | Technique | Estimator | When it helps |
|---|---|---|---|
| 1 | Antithetic | $\frac{1}{M}\sum \frac{Y_m + \tilde Y_m}{2}$ with $\tilde Y_m$ using $-Z_m$ | $\mathrm{Cov}(Y,\tilde Y)<0$, i.e. monotone payoff |
| 2 | Control variate | $\bar Y_M + \hat\beta(C^* - \bar Y^*_M)$ | $\mathrm{Corr}(Y, Y^*)$ large; $C^*$ known |
| 3 | Conditional MC | $\frac{1}{M}\sum f(X_m)$ where $f(X) = \mathbb E[Y|X]$ | $f$ analytically computable; always helps |
| 4 | Optimal $\beta$ | $\beta_{\text{opt}} = \mathrm{Cov}(Y,Y^*)/\mathrm{Var}(Y^*)$ | At optimum: $\mathrm{Var}$ shrinks by $1 - \mathrm{Corr}^2(Y,Y^*)$ |
| 5 | Geometric basket | $S_0^G = \sqrt{\prod S_0^{[j]}}$, $\sigma_G = \tfrac12\sqrt{\sigma_1^2 + 2\rho\sigma_1\sigma_2 + \sigma_2^2}$, $q_G = (\sigma_1^2+\sigma_2^2-2\rho\sigma_1\sigma_2)/8$ | Control variate for arithmetic basket |
| 6 | Mixing formula | $C = \mathbb E\,C_{BS}(\bar\sigma)$ where $\bar\sigma^2 = \tfrac1T\int V_t\,dt$ | Stoch-vol, $W_1 \perp W_2$ |
| 7 | SE for AV | std of pair averages, divided by $\sqrt M$ | Not the std of $2M$ values! |
| 8 | $r - q$ | Drift, not discount factor | Practice-Final P3 Bug 3 |
| 9 | Confidence radius | $1.96 \cdot \hat\sigma_M/\sqrt M$ | Always 1.96 for 95% |

---

*Next:* **Topic G — Practice-Final P4: Bias-Variance for MC Delta** (the $\varepsilon$ that minimizes MSE in finite-difference delta).
